# Etapa 2 - Limpieza y transformación de datos
**Proyecto 2: Analítica de Airbnb en San Francisco**

 Ingeniería de datos



In [ ]:
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

## 1. Cargamos los Datos

In [ ]:
df = pd.read_csv("listings.csv")
filas_iniciales, columnas_iniciales = df.shape
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

## 2. Columnas vacías


In [ ]:
df.isnull().sum().sort_values(ascending=False).head(30)

Hay columnas que están completamente vacías. No sirven, así que las eliminamos.

In [ ]:
df = df.dropna(axis=1, how="all")
df.shape

## 3. Escoger las columnas que sirven
Miramos todas las columnas que quedan.

In [ ]:
df.columns

Muchas no sirven para responder las preguntas de negocio: links, fotos, ids, textos largos (nombre, descripción) y columnas repetidas. Nos quedamos con las que describen el alojamiento, su ubicación, el precio, las reseñas y el anfitrión.

In [ ]:
columnas = [
    "id", "neighbourhood_cleansed", "latitude", "longitude",
    "property_type", "room_type", "accommodates", "bathrooms_text", "bedrooms", "beds", "amenities",
    "price", "minimum_nights", "availability_365",
    "number_of_reviews", "reviews_per_month",
    "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
    "review_scores_checkin", "review_scores_communication", "review_scores_location", "review_scores_value",
    "host_is_superhost", "host_identity_verified", "hosts_time_as_host_years", "host_listings_count",
    "estimated_occupancy_l365d", "estimated_revenue_l365d",
]
df = df[columnas]
df.shape

## 4. Duplicados

In [ ]:
df.duplicated().sum()

In [ ]:
df["id"].duplicated().sum()

## 5. Precio


In [ ]:
df["price"].head(10)

Se convierte la variable price a número.

In [ ]:
df["price"] = df["price"].str.replace("$", "").str.replace(",", "")
df["price"] = df["price"].astype(float)
df["price"].describe()

Hay anuncios sin precio. Como el precio es lo que queremos predecir, no lo podemos inventar: eliminamos esas filas.

In [ ]:
df["price"].isnull().sum()

In [ ]:
df = df.dropna(subset=["price"])
df.shape

## 6. Columnas de sí/no
Miramos qué valores tienen.

In [ ]:
df["host_is_superhost"].value_counts()

In [ ]:
df["host_identity_verified"].value_counts()

Vienen como `t` y `f`. Las pasamos a 1 y 0.

In [ ]:
df["host_is_superhost"] = df["host_is_superhost"].map({"t": 1, "f": 0})
df["host_identity_verified"] = df["host_identity_verified"].map({"t": 1, "f": 0})
df[["host_is_superhost", "host_identity_verified"]].head()

## 7. Baños
La columna de baños viene como texto.

In [ ]:
df["bathrooms_text"].value_counts()

Se crea una variable que cuente directamente el número de baños y otra que indique si el baño es compartido. Los casos que dicen "half-bath" (medio baño) no traen número, así que se les pone 0,5.

In [ ]:
df["bathrooms"] = pd.to_numeric(df["bathrooms_text"].str.split(" ").str[0], errors="coerce")
df.loc[df["bathrooms_text"].str.contains("half-bath", case=False, na=False), "bathrooms"] = 0.5
df["shared_bath"] = df["bathrooms_text"].str.contains("shared", case=False).astype(float)
df[df["bathrooms_text"].str.contains("half", case=False, na=False)][["bathrooms_text", "bathrooms", "shared_bath"]].head()

In [ ]:
df = df.drop(columns=["bathrooms_text"])

## 8. Amenidades


In [ ]:
df["amenities"].iloc[0]

Vemos que es una lista de todo lo que ofrece el alojamiento. Convertimos el texto en una lista para contar cuántas amenidades tiene cada anuncio (contar por comas daría de más, porque algunas amenidades traen comas en su descripción). Luego creamos columnas de sí o no para las más importantes para un usuario. Para la lavadora quitamos primero "Dishwasher" (lavavajillas), para no confundirlos.

In [ ]:
df["n_amenities"] = df["amenities"].apply(json.loads).str.len()

df["wifi"] = df["amenities"].str.contains("Wifi", case=False).astype(int)
df["kitchen"] = df["amenities"].str.contains("Kitchen", case=False).astype(int)
df["parking"] = df["amenities"].str.contains("parking on premises", case=False).astype(int)
df["washer"] = df["amenities"].str.replace("Dishwasher", "").str.contains("washer", case=False).astype(int)
df["air_conditioning"] = df["amenities"].str.contains("Air conditioning", case=False).astype(int)
df["workspace"] = df["amenities"].str.contains("Dedicated workspace", case=False).astype(int)

df[["n_amenities", "wifi", "kitchen", "parking", "washer", "air_conditioning", "workspace"]].mean()

In [ ]:
df = df.drop(columns=["amenities"])

## 9. Tipo de propiedad
Miramos cuántas categorías tiene.

In [ ]:
df["property_type"].value_counts()

Hay muchas categorías por lo cual las más frecuentes las dejamos iguales y agrupamos las categorías poco frecuentes en una categoría "Other"

In [ ]:
principales = df["property_type"].value_counts().index[:8]
df.loc[~df["property_type"].isin(principales), "property_type"] = "Other"
df["property_type"].value_counts()

In [ ]:
df["room_type"].value_counts()

In [ ]:
df["neighbourhood_cleansed"].value_counts()

## 10. Datos atípicos
Miramos el precio con un diagrama de caja.

In [ ]:
df["price"].describe()

In [ ]:
plt.boxplot(df["price"])
plt.title("Precio por noche")
plt.show()

El precio tiene valores muy altos que no son normales. Como está muy cargado a la derecha, usamos el rango intercuartílico (IQR) sobre el logaritmo del precio y quitamos lo que queda por fuera.

In [ ]:
log_precio = np.log(df["price"])
q1 = log_precio.quantile(0.25)
q3 = log_precio.quantile(0.75)
iqr = q3 - q1

minimo = np.exp(q1 - 1.5 * iqr)
maximo = np.exp(q3 + 1.5 * iqr)
print("Precio mínimo:", round(minimo, 1))
print("Precio máximo:", round(maximo, 1))

In [ ]:
df = df[(df["price"] >= minimo) & (df["price"] <= maximo)]
df.shape

In [ ]:
plt.boxplot(df["price"])
plt.title("Precio por noche (sin atípicos)")
plt.show()

Revisamos también si hay anuncios con datos raros, como más habitaciones que huéspedes.

In [ ]:
df[df["bedrooms"] > df["accommodates"]][["accommodates", "bedrooms", "beds", "room_type", "price"]]

Casi todos son habitaciones privadas para pocas personas dentro de casas con varias habitaciones, o propiedades donde el anfitrión limita el número de huéspedes. No son errores, así que **no los eliminamos**.

## 11. Columnas nuevas
- **log_price**: el logaritmo del precio, para el modelo de regresión.
- **long_stay**: 1 si el anuncio pide 30 noches o más.
- **has_reviews**: 1 si el anuncio tiene reseñas (las calificaciones faltan cuando no las tiene).
- **recommended**: 1 si el anuncio tiene al menos 5 reseñas y calificación de 4,9 o más. Con menos de 5 reseñas no tenemos suficiente información para saber si es bueno o no, por eso creamos también **can_evaluate**: los modelos de clasificación se entrenan solo con los anuncios que tienen `can_evaluate = 1`.

In [ ]:
df["log_price"] = np.log(df["price"])
df["long_stay"] = (df["minimum_nights"] >= 30).astype(int)
df["has_reviews"] = df["review_scores_rating"].notnull().astype(int)

df["can_evaluate"] = (df["number_of_reviews"] >= 5).astype(int)
df["recommended"] = ((df["can_evaluate"] == 1) & (df["review_scores_rating"] >= 4.9)).astype(int)

df[df["can_evaluate"] == 1]["recommended"].value_counts()

**¿Por qué mínimo 5 reseñas y calificación de 4,9?**
- Con pocas reseñas la calificación no es confiable: miramos qué porcentaje de los anuncios con 1 a 4 reseñas tiene un 5,0 perfecto.
- La calificación general resume bien a las demás calificaciones: miramos su correlación con ellas.
- 4,9 es aproximadamente la mediana de la calificación de los anuncios evaluables, así que las dos clases quedan casi del mismo tamaño.

In [ ]:
pocas_resenas = df[(df["number_of_reviews"] >= 1) & (df["number_of_reviews"] < 5)]
print("% de anuncios con 1 a 4 reseñas que tienen 5,0:", round((pocas_resenas["review_scores_rating"] == 5).mean() * 100, 1))

evaluables = df[df["can_evaluate"] == 1]
print("Mediana de la calificación en los evaluables:", evaluables["review_scores_rating"].median())
print("% de recomendados entre los evaluables:", round(evaluables["recommended"].mean() * 100, 1))

calificaciones = ["review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
                  "review_scores_checkin", "review_scores_communication", "review_scores_location",
                  "review_scores_value"]
evaluables[calificaciones].corr()["review_scores_rating"].round(2)

## 12. Datos de entrenamiento y prueba
Separamos de una vez los datos en entrenamiento (80%) y prueba (20%), al azar y con una semilla fija para que siempre salgan los mismos. Así todos los modelos (regresión, clasificación estadística y red neuronal) se evalúan con los mismos anuncios y se pueden comparar.

In [ ]:
prueba = df.sample(frac=0.2, random_state=42)
df["split"] = "train"
df.loc[prueba.index, "split"] = "test"
df["split"].value_counts()

## 13. Datos faltantes
Revisamos qué falta todavía.

In [ ]:
df.isnull().sum()

- Las calificaciones faltan cuando el anuncio no tiene reseñas. Ya creamos `has_reviews` para no perder esa información, y las calificaciones se llenan con la mediana.
- `reviews_per_month` se llena con 0, porque si no hay reseñas son cero.
- Habitaciones, camas, baños y noches mínimas son pocos casos: se llenan con la mediana.
- Las medianas se calculan **solo con los datos de entrenamiento** y se usan para llenar los dos grupos. Así los datos de prueba no influyen en el entrenamiento.

In [ ]:
entrenamiento = df[df["split"] == "train"]

for col in calificaciones + ["bedrooms", "beds", "bathrooms", "shared_bath", "minimum_nights"]:
    df[col] = df[col].fillna(entrenamiento[col].median())

df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

df.isnull().sum().sum()

## 14. Guardar los datos limpios
Primero guardamos la versión completa con las categorías en texto, para el análisis exploratorio y el tablero.

In [ ]:
df.info()

In [ ]:
df.to_csv("listings_limpio.csv", index=False)

Para los modelos, convertimos las variables categóricas en variables dummy y guardamos por separado los datos de entrenamiento y de prueba.

In [ ]:
df_modelo = pd.get_dummies(df, columns=["room_type", "property_type", "neighbourhood_cleansed"],
                           drop_first=True, dtype=int)
df_modelo.shape

In [ ]:
train = df_modelo[df_modelo["split"] == "train"].drop(columns=["split"])
test = df_modelo[df_modelo["split"] == "test"].drop(columns=["split"])

train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)
print("train:", train.shape, "| test:", test.shape)

## 15. Resumen

In [ ]:
print("Datos originales:", filas_iniciales, "filas x", columnas_iniciales, "columnas")
print("Datos limpios:", df.shape[0], "filas x", df.shape[1], "columnas")
print("Filas conservadas:", round(df.shape[0] / filas_iniciales * 100, 1), "%")
print("Datos faltantes:", df.isnull().sum().sum())
print("train.csv:", train.shape, "| test.csv:", test.shape)

### Notas para el equipo
- **Análisis exploratorio y tablero:** usar `listings_limpio.csv`. Para analizar calificaciones, filtrar `has_reviews == 1`, porque en los anuncios sin reseñas la calificación fue llenada con la mediana.
- **Modelos:** entrenar con `train.csv` y evaluar con `test.csv`.
- **Regresión:** usar `log_price` como variable a predecir. No usar `price`, `estimated_revenue_l365d` ni `estimated_occupancy_l365d` como predictores, porque se calculan con el precio.
- **Clasificación:** usar solo los anuncios con `can_evaluate == 1` (en train y en test) y no usar las calificaciones (`review_scores_*`) ni `has_reviews` como predictores, porque `recommended` sale de ellas.